In [ ]:
# BERT-Code (bert-base-german-cased)

In [2]:
# -*- coding: utf-8 -*-
"""
PyTorch BERT-base-cased Sentiment Analysis - COMPLETE UPGRADE!
Läuft mit transformers - viel besser als LSTM! + Timing + Plots
Binary Sentiment (0=neg, 1=pos) - Deine sentiment_Analyse.csv
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
#from torch.optim.lr_scheduler import get_linear_schedule_with_warmup

from transformers import get_linear_schedule_with_warmup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import time
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 🔥 EXTERNAL DEP: pip install transformers
from transformers import BertTokenizer, BertModel

print('🚀 BERT-base-cased SENTIMENT ANALYSIS - STATE-OF-THE-ART!')
print('─' * 80)

# CUDA Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

print('\n─'*50 + ' 1. DATA LOADING & PREP ' + '─'*50)

# Deine CSV laden (Spalte 0=label*4, Spalte 5=text)
data = pd.read_csv('sentiment_Analyse.csv', delimiter=',', encoding='utf-8')
print(f'📊 Total datapoints: {len(data)}')

y = data.iloc[:, 0].values.astype('float32') / 4  # 0.0=neg, 1.0=pos → int
y = (y > 0.5).astype(int)  # Binary: 0=neg, 1=pos
X = data.iloc[:, 5].values  # Tweets

print('First 3 examples:')
for i in range(3):
    print(f'  Label: {y[i]} | Text: {X[i][:60]}...')

# Clean text (vereinfacht für BERT - behält case!)
def clean_text(text):
    # Nur @ entfernen + basic cleanup
    words = text.split()
    words = [w for w in words if not w.startswith('@')]
    text = ' '.join(words)
    # XML tags entfernen
    text = text.replace('&ampPOOOOOLL', ' ').replace('&quotPOOOOOLL', ' ').replace('&ltPOOOOOLL', ' ').replace('&gtPOOOOOLL', ' ').replace('POOOOOLL', ' ')
    return text.strip()

X_clean = [clean_text(text) for text in X]
print('\n✅ First cleaned:')
for text in X_clean[:3]:
    print(f'  {repr(text[:60])}...')

# Train/Val/Test Split
X_temp, X_test, y_temp, y_test = train_test_split(X_clean, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

print(f'✅ Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

print('\n─'*50 + ' 2. BERT TOKENIZER & DATASETS ' + '─'*50)

# BERT-base-cased laden
model_name = 'bert-base-cased'
tokenizer = BertTokenizer.from_pretrained(model_name)
MAX_LEN = 128  # Optimal für Tweets

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            pad_to_max_length=True,
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Datasets + DataLoaders
train_ds = SentimentDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds = SentimentDataset(X_val, y_val, tokenizer, MAX_LEN)
test_ds = SentimentDataset(X_test, y_test, tokenizer, MAX_LEN)

batch_size = 32  # GPU-optimiert
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)

print('✅ DataLoaders ready! Sample batch:')
sample_batch = next(iter(train_loader))
print(f'   Shapes: input_ids={sample_batch["input_ids"].shape}, labels={sample_batch["labels"].shape}')

print('\n─'*50 + ' 3. BERT CLASSIFIER MODEL ' + '─'*50)

class BertSentiment(nn.Module):
    def __init__(self, n_classes=2):
        super(BertSentiment, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[1]  # [CLS] pooled
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)

model = BertSentiment(n_classes=2).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'💾 Total Params: {total_params:,} (BERT: ~110M)')

# Hyperparams (BERT-optimal)
EPOCHS = 4  # Weniger als LSTM nötig!
optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
loss_fn = nn.CrossEntropyLoss().to(device)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

print('\n─'*50 + ' 4. TRAINING mit EPOCH-TIMING ' + '─'*50)

def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []
    correct_preds = 0
    
    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, labels)
        
        correct_preds += torch.sum(preds == labels)
        losses.append(loss.item())
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    return correct_preds.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_preds = 0
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)
            
            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, labels)
            
            correct_preds += torch.sum(preds == labels)
            losses.append(loss.item())
    
    return correct_preds.double() / n_examples, np.mean(losses)

# Training Loop
history = defaultdict(list)
best_val_acc = 0
epoch_durations = []
total_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    
    train_acc, train_loss = train_epoch(
        model, train_loader, loss_fn, optimizer, device, scheduler, len(train_ds)
    )
    val_acc, val_loss = eval_model(model, val_loader, loss_fn, device, len(val_ds))
    
    epoch_duration = time.time() - epoch_start
    epoch_durations.append(epoch_duration)
    avg_epoch = np.mean(epoch_durations)
    
    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_bert_sentiment.pth')
    
    print(f' Epoch {epoch+1:2d}: Train={train_acc:.1%}({train_loss:.4f}) | '
          f'Val={val_acc:.1%}({val_loss:.4f}) | Best={best_val_acc:.1%} | '
          f'⏱️ {epoch_duration:.1f}s (Ø{avg_epoch:.1f}s)')

total_duration = time.time() - total_start
print(f'\n🏆 FINAL RESULTS:')
print(f'   Best Val Acc:  {best_val_acc:.1%}')
print(f'   ⏱️ Total Time:  {total_duration:.1f}s')
print(f'   ⏱️ Ø Epoch:     {np.mean(epoch_durations):.1f}s')
print(f'   💾 Saved: best_bert_sentiment.pth')

print('\n─'*50 + ' 5. TEST EVALUATION ' + '─'*50)

test_acc, test_loss = eval_model(model, test_loader, loss_fn, device, len(test_ds))
print(f'🧪 Test Acc: {test_acc:.1%} (Loss: {test_loss:.4f})')

print('\n─'*50 + ' 6. TRAINING PLOTS ' + '─'*50)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

ax1.plot(history['train_loss'], 'b-', lw=2, label='Train')
ax1.plot(history['val_loss'], 'r-', lw=2, label='Val')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], 'b-', lw=2, label='Train')
ax2.plot(history['val_acc'], 'r-', lw=2, label='Val')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3.bar(range(1, EPOCHS+1), epoch_durations, color='green', alpha=0.7)
ax3.set_title('Epoch Duration')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Time (s)')
ax3.grid(True, alpha=0.3)

cum_times = np.cumsum(epoch_durations)
ax4.plot(range(1, EPOCHS+1), cum_times, 'purple', lw=3, marker='o')
ax4.set_title('Cumulative Time')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Total Time (s)')
ax4.grid(True, alpha=0.3)

plt.suptitle('BERT Sentiment Training History')
plt.tight_layout()
plt.show()

print('\n─'*50 + ' 7. INFERENCE & PREDICT FUNCTION ' + '─'*50)

def predict_bert(texts, model, tokenizer, device):
    """Ready-to-use! List[str] → List[float] probabilities (pos)"""
    model.eval()
    results = []
    for text in texts:
        encoding = tokenizer.encode_plus(
            clean_text(text),
            add_special_tokens=True,
            max_length=MAX_LEN,
            return_token_type_ids=False,
            pad_to_max_length=True,
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )
        
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids, attention_mask)
            probs = torch.softmax(outputs, dim=1)
            pos_prob = probs[0][1].cpu().item()  # Pos (1)
        
        results.append(pos_prob)
    
    return results

# Test Cases
test_cases = [
    'bad', 'sad', 'good', 'happy', 'great',
    'I love this!', 'I hate it', 'not satisfied', 'I love Trump',
    'This is terrible service!', 'Absolutely fantastic product!'
]

print('🧪 BERT Predictions:')
preds = predict_bert(test_cases, model, tokenizer, device)
for text, score in zip(test_cases, preds):
    label = 'POS' if score > 0.5 else 'NEG'
    print(f'  {text:30s} → {score:.3f} ({label})')

print('\n🎉 BERT DONE! ~95%+ Acc erwartet (vs LSTM ~85%)')
print(f'   Load: model.load_state_dict(torch.load("best_bert_sentiment.pth"))')
print(f'   predict_bert() ready für Production!')



🚀 BERT-base-cased SENTIMENT ANALYSIS - STATE-OF-THE-ART!
────────────────────────────────────────────────────────────────────────────────
✅ Device: cuda
   GPU: NVIDIA RTX 3500 Ada Generation Laptop GPU

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─ 1. DATA LOADING & PREP ──────────────────────────────────────────────────
📊 Total datapoints: 1048572
First 3 examples:
  Label: 0 | Text: is upset that he can't update his Facebook by texting it... ...
  Label: 0 | Text: @Kenichan I dived many times for the ball. Managed to save 5...
  Label: 0 | Text: my whole body feels itchy and like its on fire ...

✅ First cleaned:
  "is upset that he can't update his Facebook by texting it... "...
  'I dived many times for the ball. Managed to save 50% The res'...
  'my whole body feels itchy and like its on fire'...
✅ Train: 754,971 | Val: 188,743 | Test: 104,858

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword arguments {'pad_to_max_length': True} not recognized.
Keyword 

✅ DataLoaders ready! Sample batch:


RuntimeError: stack expects each tensor to be equal size, but got [5] at entry 0 and [20] at entry 1

In [ ]:
# -*- coding: utf-8 -*-
"""
PyTorch Sentiment BERT - minimal change version
Original code structure preserved
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import time
from transformers import BertTokenizer, BertModel

print('🚀 PYTORCH SENTIMENT BERT (PRETRAINED)')
print('─' * 80)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')

# Hyperparameter (fast gleich)
sequence_length = 25
batch_size = 32
epochs = 5
hidden_dim = 128

print('\n' + '─'*50 + ' 1. DATA LOADING ' + '─'*50)

data = pd.read_csv('sentiment_Analyse.csv', delimiter=',', encoding='utf-8')

y = data.iloc[:, 0].values.astype('float32') / 4
X = data.iloc[:, 5].values

print(f'📊 Total datapoints: {len(X)}')

print('\n' + '─'*50 + ' 2. TEXT CLEANING ' + '─'*50)

def clean_text(text):
    words = text.split()
    words = [w for w in words if not w.startswith('@')]
    text = ' '.join(words)
    return text.lower()

X_clean = [clean_text(t) for t in X]

X_train, X_val, y_train, y_val = train_test_split(
    X_clean, y, test_size=0.2, random_state=0
)

print(f'✅ Train: {len(X_train)} | Val: {len(X_val)}')

print('\n' + '─'*50 + ' 3. TOKENIZER (BERT) ' + '─'*50)

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

class SentimentDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len):

        self.texts = texts
        self.labels = torch.FloatTensor(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()

        return input_ids, attention_mask, self.labels[idx]


train_ds = SentimentDataset(X_train, y_train, tokenizer, sequence_length)
val_ds = SentimentDataset(X_val, y_val, tokenizer, sequence_length)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

print('✅ DataLoader ready')

print('\n' + '─'*50 + ' 4. BERT MODEL ' + '─'*50)

class SentimentBERT(nn.Module):

    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-cased")

        self.fc1 = nn.Linear(768, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:,0,:]

        x = self.relu(self.fc1(cls))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x


model = SentimentBERT().to(device)

print(model)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

print('\n' + '─'*50 + ' 5. TRAINING ' + '─'*50)

best_val_acc = 0

train_losses = []
val_losses = []

train_accs = []
val_accs = []

epoch_durations = []

total_start_time = time.time()

for epoch in range(epochs):

    epoch_start = time.time()

    model.train()

    train_loss = 0
    train_correct = 0

    for input_ids, mask, labels in train_loader:

        input_ids = input_ids.to(device)
        mask = mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        preds = model(input_ids, mask).squeeze()

        loss = criterion(preds, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        train_loss += loss.item()

        train_correct += ((preds > 0.5) == (labels > 0.5)).sum().item()

    model.eval()

    val_loss = 0
    val_correct = 0

    with torch.no_grad():

        for input_ids, mask, labels in val_loader:

            input_ids = input_ids.to(device)
            mask = mask.to(device)
            labels = labels.to(device)

            preds = model(input_ids, mask).squeeze()

            loss = criterion(preds, labels)

            val_loss += loss.item()

            val_correct += ((preds > 0.5) == (labels > 0.5)).sum().item()

    train_acc = train_correct / len(train_ds)
    val_acc = val_correct / len(val_ds)

    train_losses.append(train_loss / len(train_loader))
    val_losses.append(val_loss / len(val_loader))

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc

    epoch_duration = time.time() - epoch_start
    epoch_durations.append(epoch_duration)

    print(f'Epoch {epoch+1}: '
          f'Train={train_acc:.1%} '
          f'Val={val_acc:.1%} '
          f'Best={best_val_acc:.1%} '
          f'⏱ {epoch_duration:.1f}s')

print('\n🏆 FINAL RESULTS')
print(f'Best Val Acc: {best_val_acc:.1%}')

🚀 PYTORCH SENTIMENT BERT (PRETRAINED)
────────────────────────────────────────────────────────────────────────────────
✅ Device: cuda

────────────────────────────────────────────────── 1. DATA LOADING ──────────────────────────────────────────────────
📊 Total datapoints: 1048572

────────────────────────────────────────────────── 2. TEXT CLEANING ──────────────────────────────────────────────────
✅ Train: 838857 | Val: 209715

────────────────────────────────────────────────── 3. TOKENIZER (BERT) ──────────────────────────────────────────────────
✅ DataLoader ready

────────────────────────────────────────────────── 4. BERT MODEL ──────────────────────────────────────────────────
SentimentBERT(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (